<a href="https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/Solved_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## Unit of Analysis:  
One row equals one specific piece of content (URL) for one specific client.
## Time Window:
We are extracting data from a mid-panel month partition (month=2026-03). For our features (the inputs), we will use a trailing 90-day lookback window anchored in March. For our label (the target we are guessing), we will look at the subsequent 30-day window to see what actually happened to the traffic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

##Context (Metadata):
client_id, content_id. These are identifiers used strictly for grouping and splitting data, never as predictive features.

##Features (The Inputs):
content_age_days, trailing_30d_impressions, avg_position, ctr_30d. These are historical, observable facts knowable at the exact moment we make the decision to refresh the content.

##Label (The Target):
is_declining_label (Binary: 1 = declining, 0 = stable/growing).

##Excluded (The Leaks & Holes):

trend_direction and trend_pct must be strictly excluded because they are used to mathematically calculate the label. Including them is data leakage (cheating).

Any rows where ga4_data_available is FALSE. Google backfills missing analytics with zeros. We exclude these so the model does not mistake a broken software connection for zero human engagement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

##ontent_age_days:
Knowable at the decision moment because the original publication date is locked in the dimension table before our anchor date.

##gsc_impressions (trailing 30d):
Knowable at the decision moment because it aggregates historical search console views that have already occurred.

##gsc_avg_position:
Knowable at the decision moment because ranking snapshots are recorded daily prior to the prediction window.

##gsc_clicks (trailing 30d):
Knowable at the decision moment because historical clicks have already resolved.

##ga4_sessions (trailing 30d):
Knowable at the decision moment because it strictly filters on ga4_data_available = TRUE for days preceding the decision point.

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Securely load your read token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

print("Connecting to warehouse and loading March 2026 partition...")
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    split="train",
    token=hf_token
)
df = dataset.to_pandas()

print("-" * 40)
# Query 1: Verify the Date Span and Row Count
min_date = df['report_date'].min()
max_date = df['report_date'].max()
print(f"Date Span: {min_date} to {max_date}")
print(f"Total Rows in March partition: {len(df):,}")

# Query 2: Verify the Grain
# We use the hash_id columns we just discovered!
grain_check = df.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size()
print(f"Max rows per report_date/client/content: {grain_check.max()} (If 1, grain is perfectly clean)")

# Query 3: Verify Missing Values (The IS TRUE filter)
available_df = df[df['ga4_data_available'] == True]
survival_rate = len(available_df) / len(df)
print(f"GA4 Availability: {len(available_df):,} rows survived the IS TRUE filter ({survival_rate:.1%} of data is usable)")

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

print("\n--- THE LEAKAGE TRAP ---")
# We take a sample of the real columns to build a quick feature frame
feature_frame = available_df[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']].dropna().sample(10000, random_state=42)

# We generate a simulated target (1 = declining, 0 = stable) to test our math safely
feature_frame['is_declining_label'] = np.random.randint(0, 2, size=len(feature_frame))

# We derive a feature directly from the target. This is data leakage.
feature_frame['trap_trend_direction'] = feature_frame['is_declining_label'] * 0.95 + np.random.rand(len(feature_frame)) * 0.05

print("1. Running WITH the trap (Leaked Feature)")
X_leak = feature_frame[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'trap_trend_direction']]
y = feature_frame['is_declining_label']
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)
leak_score = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy: {leak_score:.1%} (Dangerously perfect - the model is cheating)\n")

print("2. Running HONESTLY (Removed the trap)")
X_honest = feature_frame[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)

model.fit(X_train_h, y_train_h)
honest_score = accuracy_score(y_test_h, model.predict(X_test_h))
print(f"Accuracy: {honest_score:.1%} (Real, honest signal)")

Connecting to warehouse and loading March 2026 partition...
----------------------------------------
Date Span: 2026-03-01 to 2026-03-31
Total Rows in March partition: 9,841,378
Max rows per report_date/client/content: 1 (If 1, grain is perfectly clean)
GA4 Availability: 413,966 rows survived the IS TRUE filter (4.2% of data is usable)

--- THE LEAKAGE TRAP ---
1. Running WITH the trap (Leaked Feature)
Accuracy: 100.0% (Dangerously perfect - the model is cheating)

2. Running HONESTLY (Removed the trap)
Accuracy: 48.5% (Real, honest signal)


## 4. Data limits

##Limitation:
Client history depths differ wildly. By enforcing a strict filter on ga4_data_available == TRUE and requiring a trailing historical window to generate features, we will systematically drop newer clients from the training set. This biases the model toward older, established clients and limits its ability to accurately score content for newly onboarded sites.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.